# 🔍 Bloque 3 — Sesión 1: Valores ausentes y valores atípicos

**Módulo:** Depuración, limpieza y Clasificación de Datos (5109)  
**Unidad Didáctica:** UD1 — Análisis exploratorio de datos  
**Duración estimada:** ~3 horas.

---

### Contenidos de esta sesión
- **3.1** Valores ausentes (*missing values*): tipos (MCAR, MAR, MNAR) y detección con `isnull`, librería `missingno`
- **3.2** Valores atípicos (*outliers*): detección visual (boxplot) y estadística (IQR, z-score)

### Criterio de evaluación asociado
> **(CE b)** Se han detectado datos incorrectamente introducidos, ausencia de valores, una mala codificación de las variables o valores atípicos.

In [ ]:
# Librerías necesarias (ejecutar una sola vez)
import subprocess, sys
for pkg in ['missingno']:
    subprocess.run([sys.executable, '-m', 'pip', 'install', pkg, '-q'], capture_output=True)

import pandas as pd
import numpy as np
import seaborn as sns
import matplotlib.pyplot as plt
import missingno as msno
from scipy import stats

sns.set_theme(style='whitegrid', palette='muted')
plt.rcParams['figure.dpi'] = 100

# Datasets
titanic  = sns.load_dataset('titanic')
penguins = sns.load_dataset('penguins')

print('✅ Librerías y datasets cargados correctamente')

---
## 3.1 Valores ausentes (*missing values*)

Un valor ausente es una celda sin dato. En pandas se representa como `NaN` (*Not a Number*) para numéricos o `None` para objetos. Su presencia es casi inevitable en datos reales y su tratamiento incorrecto puede sesgar gravemente cualquier modelo.

### Tipos de ausencia (taxonomía de Rubin)

Antes de decidir cómo tratar un ausente, hay que entender **por qué falta**:

| Tipo | Nombre completo | Descripción | Ejemplo |
|------|-----------------|-------------|----------|
| **MCAR** | *Missing Completely At Random* | La ausencia no depende ni del valor perdido ni de otras variables. Es aleatoria pura. | Un sensor falla aleatoriamente y no registra temperatura en algunos instantes. |
| **MAR** | *Missing At Random* | La ausencia depende de otras variables observadas, pero no del valor perdido en sí. | Los hombres responden menos a la pregunta de peso en encuestas (depende del sexo, que sí tenemos). |
| **MNAR** | *Missing Not At Random* | La ausencia depende del propio valor no observado. Es el caso más problemático. | Los pacientes con valores de colesterol muy altos evitan hacerse análisis. |

> ⚠️ Solo los ausentes **MCAR** pueden imputarse o eliminarse sin introducir sesgo. Los **MAR** requieren técnicas de imputación que usen las variables relacionadas. Los **MNAR** son los más difíciles: la imputación simple siempre introduce sesgo.

### 📌 Ejemplo 1 — Detección básica con `isnull` y `isna`

In [ ]:
# isnull() / isna() devuelven un DataFrame booleano: True donde hay NaN
# (isnull e isna son alias, hacen lo mismo)

print('Máscara de ausentes (primeras 5 filas):')
print(titanic.isnull().head())

# Conteo de ausentes por columna
ausentes_col = titanic.isnull().sum()
pct_col      = titanic.isnull().mean() * 100

resumen = pd.DataFrame({
    'ausentes': ausentes_col,
    'porcentaje': pct_col.round(1)
})
resumen = resumen[resumen['ausentes'] > 0].sort_values('porcentaje', ascending=False)

print('\nColumnas con valores ausentes:')
print(resumen)

In [ ]:
# También podemos analizar ausentes por fila
ausentes_fila = titanic.isnull().sum(axis=1)

print('Distribución de ausentes por fila (nº de celdas vacías por registro):')
print(ausentes_fila.value_counts().sort_index())

# Filas con más de 1 ausente
filas_problema = titanic[ausentes_fila > 1]
print(f'\nFilas con más de 1 valor ausente: {len(filas_problema)} ({len(filas_problema)/len(titanic)*100:.1f}%)')

### 📌 Ejemplo 2 — Visualización con `missingno`

In [ ]:
# msno.matrix: muestra la presencia/ausencia de cada celda
# Las líneas blancas indican valores ausentes
# La sparkline de la derecha muestra cuántos valores hay en cada fila
fig, ax = plt.subplots(figsize=(12, 5))
msno.matrix(titanic, ax=ax, sparkline=True, fontsize=9, color=(0.3, 0.5, 0.7))
ax.set_title('Mapa de presencia/ausencia — Titanic', fontsize=12, pad=10)
plt.tight_layout()
plt.show()

print('💡 Las columnas "age" y "deck" son las más afectadas.')
print('   La sparkline revela si los ausentes se concentran en ciertas zonas del dataset.')

In [ ]:
# msno.bar: muestra el % de completitud por columna como barras
fig, ax = plt.subplots(figsize=(12, 4))
msno.bar(titanic, ax=ax, fontsize=9, color='steelblue')
ax.set_title('Completitud por columna — Titanic', fontsize=12, pad=10)
plt.tight_layout()
plt.show()

In [ ]:
# msno.heatmap: correlación de ausencia entre pares de columnas
# Un valor cercano a 1 significa que si una columna tiene NaN, la otra también lo tiene
# Ayuda a detectar si los ausentes son MAR (relacionados entre sí)
fig, ax = plt.subplots(figsize=(9, 5))
msno.heatmap(titanic, ax=ax, fontsize=9, cmap='RdBu')
ax.set_title('Correlación de ausencia entre columnas — Titanic', fontsize=12, pad=10)
plt.tight_layout()
plt.show()

print('💡 Si dos columnas tienen correlación de ausencia alta → los NaN no son MCAR.')
print('   Indicio de que puede existir un patrón sistemático (MAR o MNAR).')

### 📌 Ejemplo 3 — Diagnóstico del tipo de ausencia (MCAR / MAR / MNAR)

In [ ]:
# Analizamos si la ausencia de 'age' en el Titanic es aleatoria o sistemática
# Estrategia: comparar otras variables según si 'age' está o no ausente

titanic['age_ausente'] = titanic['age'].isnull()

# ¿Hay diferencia en la tasa de supervivencia según si falta la edad?
print('Tasa de supervivencia según ausencia de edad:')
print(titanic.groupby('age_ausente')['survived'].mean().round(3))

# ¿Hay diferencia en la clase según si falta la edad?
print('\nDistribución de clase según ausencia de edad:')
print(pd.crosstab(titanic['age_ausente'], titanic['pclass'], normalize='index').round(3))

# Eliminamos la columna auxiliar
titanic.drop(columns='age_ausente', inplace=True)

print('\n💡 Si la supervivencia o la clase difieren entre grupos → la ausencia NO es MCAR.')
print('   La edad podría ser MAR (depende de la clase u otros factores registrados).')

In [ ]:
# Visualización del diagnóstico
fig, axes = plt.subplots(1, 2, figsize=(13, 4))

# Distribución de 'fare' según si 'age' está o no ausente
titanic['age_ausente'] = titanic['age'].isnull().map({True: 'Age ausente', False: 'Age presente'})
sns.boxplot(data=titanic, x='age_ausente', y='fare', palette='pastel',
            hue='age_ausente', legend=False, ax=axes[0])
axes[0].set_title('Fare según presencia de Age')
axes[0].set_xlabel('')
axes[0].set_ylabel('Precio del billete (£)')

# Proporción de ausencia de age por clase
pct_ausente_clase = titanic.groupby('pclass')['age'].apply(lambda x: x.isnull().mean() * 100)
axes[1].bar(['1ª clase', '2ª clase', '3ª clase'], pct_ausente_clase.values,
            color=['#4C72B0', '#55A868', '#C44E52'], edgecolor='white')
axes[1].set_title('% de Age ausente por clase')
axes[1].set_ylabel('% de ausentes')

titanic.drop(columns='age_ausente', inplace=True)
plt.tight_layout()
plt.show()

print('💡 Los pasajeros de 3ª clase tienen más edad ausente → probable MAR (depende de la clase).')

### 📌 Ejemplo 4 — Dataset sintético con los tres tipos de ausencia

In [ ]:
# Creamos un dataset que ilustra los tres tipos de ausencia
np.random.seed(42)
n = 200

df = pd.DataFrame({
    'edad':    np.random.randint(18, 65, n),
    'ingresos': np.random.normal(30000, 10000, n).round(0),
    'satisfaccion': np.random.choice([1, 2, 3, 4, 5], n),
    'genero':  np.random.choice(['M', 'F'], n)
})

# MCAR: eliminamos aleatoriamente el 10% de 'satisfaccion'
idx_mcar = np.random.choice(n, size=int(n * 0.1), replace=False)
df.loc[idx_mcar, 'satisfaccion'] = np.nan

# MAR: las mujeres responden menos a 'ingresos'
idx_mar = df[df['genero'] == 'F'].sample(frac=0.35, random_state=42).index
df.loc[idx_mar, 'ingresos'] = np.nan

# MNAR: las personas mayores de 55 no informan su edad
df.loc[df['edad'] > 55, 'edad'] = np.nan

print('Resumen de ausentes en el dataset sintético:')
for col in df.columns:
    n_aus = df[col].isnull().sum()
    if n_aus > 0:
        print(f'  {col:15s}: {n_aus:3d} ausentes ({n_aus/n*100:.1f}%)')

print('\nDiagnóstico esperado:')
print('  satisfaccion → MCAR (ausencia aleatoria, no depende de nada)')
print('  ingresos     → MAR  (ausencia depende del género, variable observada)')
print('  edad         → MNAR (ausencia depende del propio valor de edad)')

---
### 🔧 Ejercicio 1 — Diagnóstico de ausentes en Penguins

El dataset `penguins` tiene valores ausentes en varias columnas. Diagnostica su tipo y visualízalos.

In [ ]:
# TAREA 1: ¿Cuántos ausentes hay por columna y en qué porcentaje?
# print(penguins.isnull().sum())
# print(penguins.isnull().mean().round(3) * 100)

# TAREA 2: Usa msno.matrix y msno.heatmap para visualizar el patrón de ausencia
# msno.matrix(penguins)
# plt.show()

# TAREA 3: Analiza si la ausencia de 'bill_length_mm' depende de la especie o del sexo
# (pista: crea columna auxiliar 'bill_ausente', luego agrupa y compara)
# ...

# TAREA 4: ¿La ausencia de 'sex' está relacionada con la especie?
# print(pd.crosstab(penguins['sex'].isnull(), penguins['species']))

# TAREA 5 (reflexión): A partir del análisis, ¿qué tipo de ausencia crees que tiene
#                      cada columna afectada? Justifica tu respuesta.
# Respuesta: ...

print('Completa las tareas comentadas arriba.')

---
## 3.2 Valores atípicos (*outliers*)

Un valor atípico es una observación que se aleja significativamente del resto. Pueden ser:
- **Error de medición o entrada**: un peso registrado como 1200 kg en lugar de 120 kg.
- **Valor real pero extremo**: el precio de un billete de primera clase en el Titanic.
- **Valor de una distribución diferente**: un fraude entre transacciones normales.

> ⚠️ No siempre hay que eliminar un outlier. Primero hay que entender **por qué** existe. En detección de fraude, los outliers son precisamente lo que buscamos.

### Métodos de detección

| Método | Fórmula / Herramienta | Cuándo usarlo |
|--------|-----------------------|---------------|
| **Boxplot visual** | `sns.boxplot` | Primera exploración rápida |
| **Criterio IQR** | Fuera de `[Q1 − 1.5·IQR, Q3 + 1.5·IQR]` | Distribuciones asimétricas, robusto a outliers |
| **Z-score** | `(x − μ) / σ`, outlier si `|z| > 3` | Distribuciones aproximadamente normales |
| **Z-score modificado** | Basado en la mediana y MAD, más robusto | Cuando hay muchos outliers que sesgan la media |

### 📌 Ejemplo 5 — Detección visual con boxplot

In [ ]:
# Creamos un dataset con outliers conocidos para comparar métodos
np.random.seed(0)
datos_normales = np.random.normal(loc=50, scale=10, size=200)
outliers_conocidos = np.array([120, 130, -20, -30, 140])
datos_con_outliers = np.concatenate([datos_normales, outliers_conocidos])

df_demo = pd.DataFrame({'valor': datos_con_outliers})

fig, axes = plt.subplots(1, 2, figsize=(13, 4))

# Histograma
sns.histplot(df_demo['valor'], bins=30, ax=axes[0], color='steelblue', kde=True)
axes[0].set_title('Histograma con outliers')
axes[0].set_xlabel('Valor')
axes[0].axvline(df_demo['valor'].mean(),   color='red',    linestyle='--', label='Media')
axes[0].axvline(df_demo['valor'].median(), color='orange', linestyle='--', label='Mediana')
axes[0].legend()

# Boxplot
sns.boxplot(y=df_demo['valor'], ax=axes[1], color='lightsalmon')
axes[1].set_title('Boxplot — puntos = outliers potenciales')
axes[1].set_ylabel('Valor')

plt.tight_layout()
plt.show()

print(f'Media: {df_demo["valor"].mean():.2f}  |  Mediana: {df_demo["valor"].median():.2f}')
print('💡 La media se aleja de la mediana por efecto de los outliers.')

### 📌 Ejemplo 6 — Detección con el criterio IQR

In [ ]:
def detectar_outliers_iqr(df, columna, factor=1.5):
    """
    Detecta outliers usando el criterio IQR.
    Devuelve un diccionario con los límites y los índices afectados.
    """
    serie = df[columna].dropna()
    q1  = serie.quantile(0.25)
    q3  = serie.quantile(0.75)
    iqr = q3 - q1
    lim_inf = q1 - factor * iqr
    lim_sup = q3 + factor * iqr
    mascara = (df[columna] < lim_inf) | (df[columna] > lim_sup)
    return {
        'q1': q1, 'q3': q3, 'iqr': iqr,
        'lim_inf': lim_inf, 'lim_sup': lim_sup,
        'n_outliers': mascara.sum(),
        'indices': df[mascara].index.tolist(),
        'valores': df.loc[mascara, columna].tolist()
    }

# Aplicamos al dataset de demostración
resultado = detectar_outliers_iqr(df_demo, 'valor')
print(f'Q1 = {resultado["q1"]:.2f} | Q3 = {resultado["q3"]:.2f} | IQR = {resultado["iqr"]:.2f}')
print(f'Límite inferior: {resultado["lim_inf"]:.2f}')
print(f'Límite superior: {resultado["lim_sup"]:.2f}')
print(f'\nOutliers detectados: {resultado["n_outliers"]}')
print(f'Valores: {sorted(resultado["valores"])}')

# Ahora aplicamos al Titanic
print('\n── Titanic: outliers en "fare" ──────────────────────────')
res_fare = detectar_outliers_iqr(titanic, 'fare')
print(f'Límites: [{res_fare["lim_inf"]:.2f}, {res_fare["lim_sup"]:.2f}]')
print(f'Outliers: {res_fare["n_outliers"]} ({res_fare["n_outliers"]/len(titanic)*100:.1f}%)')
print(f'Rango de valores atípicos: {min(res_fare["valores"]):.2f} — {max(res_fare["valores"]):.2f} £')

In [ ]:
# Visualizamos los límites IQR sobre el boxplot
fig, ax = plt.subplots(figsize=(10, 4))

fare_clean = titanic['fare'].dropna()
res = detectar_outliers_iqr(titanic, 'fare')

ax.scatter(fare_clean, np.zeros(len(fare_clean)), alpha=0.3, s=20, color='steelblue', label='Datos')
ax.scatter(
    res['valores'], np.zeros(len(res['valores'])),
    color='red', s=50, zorder=5, label='Outliers IQR'
)
ax.axvline(res['lim_sup'], color='red', linestyle='--', alpha=0.7, label=f'Límite sup. = {res["lim_sup"]:.1f} £')
ax.axvline(res['q1'], color='green', linestyle=':', alpha=0.7, label=f'Q1 = {res["q1"]:.1f} £')
ax.axvline(res['q3'], color='green', linestyle=':', alpha=0.7, label=f'Q3 = {res["q3"]:.1f} £')
ax.set_title('Fare — Detección de outliers por criterio IQR')
ax.set_xlabel('Precio del billete (£)')
ax.set_yticks([])
ax.legend(loc='upper right', fontsize=8)
plt.tight_layout()
plt.show()

### 📌 Ejemplo 7 — Detección con Z-score y Z-score modificado

In [ ]:
def detectar_outliers_zscore(df, columna, umbral=3.0):
    """Detecta outliers con el z-score estándar (|z| > umbral)."""
    serie = df[columna].dropna()
    z = np.abs(stats.zscore(serie))
    outlier_idx = serie[z > umbral].index
    return df.loc[outlier_idx, columna]

def detectar_outliers_zscore_modificado(df, columna, umbral=3.5):
    """Detecta outliers con el z-score modificado basado en la mediana (MAD)."""
    serie = df[columna].dropna()
    mediana = serie.median()
    mad = np.abs(serie - mediana).median()
    z_mod = 0.6745 * np.abs(serie - mediana) / mad
    outlier_idx = z_mod[z_mod > umbral].index
    return df.loc[outlier_idx, columna]

# Comparamos los tres métodos en la columna 'fare'
out_iqr    = titanic.loc[res_fare['indices'], 'fare']
out_z      = detectar_outliers_zscore(titanic, 'fare')
out_z_mod  = detectar_outliers_zscore_modificado(titanic, 'fare')

print('Comparación de métodos — outliers en "fare" del Titanic:')
print(f'  IQR (×1.5)           : {len(out_iqr):3d} outliers | rango: {out_iqr.min():.1f} — {out_iqr.max():.1f} £')
print(f'  Z-score (|z|>3)      : {len(out_z):3d} outliers | rango: {out_z.min():.1f} — {out_z.max():.1f} £')
print(f'  Z-score modificado   : {len(out_z_mod):3d} outliers | rango: {out_z_mod.min():.1f} — {out_z_mod.max():.1f} £')

print('\n💡 El criterio IQR detecta más outliers porque la distribución de fare')
print('   es muy asimétrica (cola larga derecha). El z-score asume normalidad')
print('   y es menos sensible en distribuciones sesgadas.')

### 📌 Ejemplo 8 — Detección multivariante con scatter plot

In [ ]:
# Un valor puede no ser outlier en cada variable por separado,
# pero sí en la combinación de ambas
penguins_clean = penguins.dropna(subset=['bill_length_mm', 'body_mass_g'])

# Detectamos outliers IQR en cada dimensión
out_bill = detectar_outliers_iqr(penguins_clean, 'bill_length_mm')
out_mass = detectar_outliers_iqr(penguins_clean, 'body_mass_g')
idx_outliers = set(out_bill['indices']) | set(out_mass['indices'])

colores = penguins_clean.index.map(
    lambda i: 'red' if i in idx_outliers else 'steelblue'
)

fig, ax = plt.subplots(figsize=(9, 5))
ax.scatter(
    penguins_clean['bill_length_mm'],
    penguins_clean['body_mass_g'],
    c=colores, alpha=0.6, s=40, edgecolors='none'
)
ax.set_title('Outliers multivariantes — Penguins (rojo = outlier en alguna dimensión)')
ax.set_xlabel('Longitud del pico (mm)')
ax.set_ylabel('Masa corporal (g)')
plt.tight_layout()
plt.show()

print(f'Outliers detectados: {len(idx_outliers)} ({len(idx_outliers)/len(penguins_clean)*100:.1f}%)')

---
### 🔧 Ejercicio 2 — Detección de outliers en el dataset Diamonds

Aplica los métodos aprendidos sobre el dataset `diamonds` de seaborn.

In [ ]:
diamonds = sns.load_dataset('diamonds')
cols_num = ['carat', 'depth', 'table', 'price', 'x', 'y', 'z']

# TAREA 1: Aplica la función detectar_outliers_iqr a cada columna numérica
#           y muestra cuántos outliers tiene cada una
# for col in cols_num:
#     res = detectar_outliers_iqr(diamonds, col)
#     print(f'{col:8s}: {res["n_outliers"]:5d} outliers ({res["n_outliers"]/len(diamonds)*100:.2f}%)')

# TAREA 2: Dibuja boxplots de todas las columnas numéricas en un grid 2×4
# ...

# TAREA 3: Las columnas 'x', 'y', 'z' son dimensiones físicas (mm).
#           ¿Hay valores 0 en ellas? Un diamante no puede tener dimensión 0 → error.
#           ¿Cuántas filas tienen al menos un 0 en x, y o z?
# ...

# TAREA 4: Compara el criterio IQR con el z-score para la columna 'price'
#           ¿Qué método detecta más outliers? ¿Por qué?
# ...

# TAREA 5 (reflexión): ¿Los outliers de 'price' son errores de datos o valores legítimos?
#                      ¿Cómo lo sabrías?
# Respuesta: ...

print('Completa las tareas comentadas arriba.')

---
### 🔧 Ejercicio 3 — Informe de calidad: ausentes + outliers

Crea una función `informe_calidad(df, nombre)` que combine el análisis de ausentes y outliers en un único resumen.

In [ ]:
def informe_calidad(df, nombre='Dataset'):
    """
    Genera un informe de calidad con:
    - Dimensiones y tipos
    - Valores ausentes por columna (%)
    - Outliers IQR por columna numérica (%)
    """
    sep = '═' * 55
    print(f'\n{sep}')
    print(f'  INFORME DE CALIDAD: {nombre}')
    print(sep)
    print(f'  Dimensiones: {df.shape[0]:,} filas × {df.shape[1]} columnas')

    # --- BLOQUE 1: Ausentes ---
    print('\n  VALORES AUSENTES')
    print('  ' + '─' * 50)
    ausentes = df.isnull().sum()
    ausentes = ausentes[ausentes > 0]
    if ausentes.empty:
        print('  ✅ Sin valores ausentes.')
    else:
        for col, n in ausentes.sort_values(ascending=False).items():
            barra = '█' * int(n / len(df) * 20)
            print(f'  {col:20s}: {n:4d} ({n/len(df)*100:5.1f}%) {barra}')

    # --- BLOQUE 2: Outliers (solo numéricas) ---
    print('\n  OUTLIERS (criterio IQR × 1.5)')
    print('  ' + '─' * 50)
    num_cols = df.select_dtypes(include='number').columns
    for col in num_cols:
        res = detectar_outliers_iqr(df, col)
        if res['n_outliers'] > 0:
            barra = '█' * int(res['n_outliers'] / len(df) * 20)
            print(f'  {col:20s}: {res["n_outliers"]:4d} ({res["n_outliers"]/len(df)*100:5.1f}%) {barra}')
    print(sep)

# Probamos la función con el Titanic y los Penguins
# (completa las llamadas)
# informe_calidad(titanic, 'Titanic')
# informe_calidad(penguins, 'Penguins')

print('Implementa las llamadas a informe_calidad para Titanic y Penguins.')

---
## 📝 Resumen de la sesión

| Concepto | Clave |
|----------|-------|
| MCAR | Ausencia aleatoria pura. Imputable sin sesgo. |
| MAR | Ausencia depende de otras variables observadas. Imputar usando esas variables. |
| MNAR | Ausencia depende del propio valor. La imputación simple introduce sesgo. |
| `isnull().sum()` | Conteo de ausentes por columna. |
| `missingno` | Librería especializada: `matrix`, `bar`, `heatmap` de ausentes. |
| Criterio IQR | Robusto y no asume normalidad. Outlier si `x < Q1−1.5·IQR` o `x > Q3+1.5·IQR`. |
| Z-score | Asume normalidad. Outlier si `|z| > 3`. Sensible a la presencia de outliers. |
| Z-score modificado | Basado en mediana y MAD. Más robusto en distribuciones sesgadas. |

### 🔜 Próxima sesión
**Bloque 3 — Sesión 2:** Datos incorrectos (errores tipográficos, duplicados, tipos mal asignados), mala codificación de variables y evaluación de cobertura.